In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MODEL.pkl')
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [ ]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

### Top EVs for single bets

In [7]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 200) & (usData['ODDS'] >= -200)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, edge_threshold=0.20, stake=10, 
                     variance_inflation=1.1, distribution_type='t', stat_col='PTS', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV%', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV%','KELLY_FRACTION','SIGMA FLAG']]
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head()

Processing single bets with single model...


,NAME,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV%,KELLY_FRACTION,SIGMA FLAG
0,John Konchar,4.5,7.91,Over,100,1,5.02,0.502,Med
1,Shai Gilgeous-Alexander,32.5,30.35,Under,-118,0,2.61,0.308,Med
2,Shai Gilgeous-Alexander,32.5,30.35,Under,-121,0,2.47,0.299,Med
3,Shai Gilgeous-Alexander,34.5,30.35,Under,-200,0,2.11,0.422,Med
4,Shai Gilgeous-Alexander,33.5,30.35,Under,-170,0,1.93,0.328,Med


## Top EVs for 2 leg bets

### Underdog picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
# underdogPairs = underdogPairs[
#     underdogPairs[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
# ].sort_values(by='EV%', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Shai Gilgeous-Alexander,Zion Williamson,31.5,20.5,30.35,20.82,under,over,0,-0.17,0.000,Med,High
1,Shai Gilgeous-Alexander,Jordan Poole,31.5,15.5,30.35,15.65,under,over,0,-0.19,0.000,Med,High
2,Shai Gilgeous-Alexander,Jose Alvarado,31.5,4.5,30.35,8.49,under,over,0,0.15,0.074,Med,High
3,Shai Gilgeous-Alexander,Trey Murphy III,31.5,16.5,30.35,13.56,under,under,0,0.02,0.008,Med,High
4,Shai Gilgeous-Alexander,Saddiq Bey,31.5,7.5,30.35,7.11,under,under,0,-0.14,0.000,Med,High


### Prizepicks picks

In [11]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=100, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
# pairsPrizepicks = pairsPrizepicks[
#     pairsPrizepicks[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
# ].sort_values(by='EV%', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Shai Gilgeous-Alexander,Zion Williamson,31.5,21.5,30.35,20.82,under,under,0,-0.13,0.000,Med,High
1,Shai Gilgeous-Alexander,Trey Murphy III,31.5,17.5,30.35,13.56,under,under,0,0.08,0.038,Med,High
2,Shai Gilgeous-Alexander,Jordan Poole,31.5,15.5,30.35,15.65,under,over,0,-0.19,0.000,Med,High
3,Shai Gilgeous-Alexander,Jeremiah Fears,31.5,12.5,30.35,11.01,under,under,0,-0.08,0.000,Med,High
4,Shai Gilgeous-Alexander,Saddiq Bey,31.5,7.5,30.35,7.11,under,under,0,-0.14,0.000,Med,High


## 3 leg parlay

### Underdog picks

In [10]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=100, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg
underdogTrios = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV%', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Jose Alvarado,Donovan Mitchell,Jalen Smith,4.5,26.5,7.5,14.42,19.37,13.13,over,under,over,1,3.57,0.714,Low,Low,Low
1,Jose Alvarado,Saddiq Bey,Donovan Mitchell,4.5,7.5,26.5,14.42,15.18,19.37,over,over,under,1,3.56,0.712,Low,Low,Low
2,Saddiq Bey,Donovan Mitchell,Jalen Smith,7.5,26.5,7.5,15.18,19.37,13.13,over,under,over,1,3.50,0.699,Low,Low,Low
3,Jose Alvarado,Saddiq Bey,Jalen Smith,4.5,7.5,7.5,14.42,15.18,13.13,over,over,over,1,3.50,0.699,Low,Low,Low
4,Jose Alvarado,Jeremiah Fears,Donovan Mitchell,4.5,11.5,26.5,14.42,17.96,19.37,over,over,under,1,3.45,0.690,Low,Low,Low


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg  
triosPrizepicks = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV%', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Jose Alvarado,Donovan Mitchell,Jalen Smith,4.5,26.5,7.5,14.42,19.37,13.13,over,under,over,1,3.62,0.723,Low,Low,Low
1,Saddiq Bey,Jose Alvarado,Donovan Mitchell,7.5,4.5,26.5,15.18,14.42,19.37,over,over,under,1,3.60,0.719,Low,Low,Low
2,Saddiq Bey,Jose Alvarado,Jalen Smith,7.5,4.5,7.5,15.18,14.42,13.13,over,over,over,0,3.51,0.702,Low,Low,Low
3,Saddiq Bey,Donovan Mitchell,Jalen Smith,7.5,26.5,7.5,15.18,19.37,13.13,over,under,over,0,3.51,0.703,Low,Low,Low
4,Jose Alvarado,Donovan Mitchell,Brandon Ingram,4.5,26.5,19.5,14.42,19.37,24.47,over,under,over,0,3.43,0.685,Low,Low,Low


In [12]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
190,PrizePicks,player_points,Shai Gilgeous-Alexander,Over,31.5,-137,2025-11-02,2025-11-02T19:09:04Z
192,PrizePicks,player_points,Zion Williamson,Over,21.5,-137,2025-11-02,2025-11-02T19:09:04Z
194,PrizePicks,player_points,Trey Murphy III,Over,17.5,-137,2025-11-02,2025-11-02T19:09:04Z
196,PrizePicks,player_points,Aaron Wiggins,Over,15.5,-137,2025-11-02,2025-11-02T19:09:04Z
198,PrizePicks,player_points,Ajay Mitchell,Over,15.5,-137,2025-11-02,2025-11-02T19:09:04Z
...,...,...,...,...,...,...,...,...
2901,PrizePicks,player_blocks_steals,Austin Reaves,Over,1.5,-137,2025-11-03,2025-11-02T19:09:53Z
2903,PrizePicks,player_blocks_steals,Luka Doncic,Over,1.5,-137,2025-11-03,2025-11-02T19:09:53Z
2905,PrizePicks,player_blocks_steals,Andrew Wiggins,Over,1.5,-137,2025-11-03,2025-11-02T19:09:53Z
2907,PrizePicks,player_blocks_steals,Bam Adebayo,Over,1.5,-137,2025-11-03,2025-11-02T19:09:53Z


In [13]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 102 records for player_points to player_points.csv
Saved 63 records for player_rebounds to player_rebounds.csv
Saved 38 records for player_assists to player_assists.csv
Saved 14 records for player_threes to player_threes.csv
Saved 8 records for player_blocks to player_blocks.csv
Saved 17 records for player_steals to player_steals.csv
Saved 49 records for player_field_goals to player_field_goals.csv
Saved 31 records for player_frees_made to player_frees_made.csv
Saved 21 records for player_frees_attempts to player_frees_attempts.csv
Saved 99 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 97 records for player_points_rebounds to player_points_rebounds.csv
Saved 88 records for player_points_assists to player_points_assists.csv
Saved 72 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 24 records for player_turnovers to player_turnovers.csv
Saved 19 records for player_blocks_steals to player_blocks_steals.csv

All categor

In [ ]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")